# 📷 Real-Time Sign Language Detection
ใช้ `best_model.keras` จาก baselineCNN พร้อม OpenCV เพื่อ detect ภาษามือแบบ real-time ผ่านกล้อง

**วิธีใช้:**
1. Run ทุก cell ตามลำดับ
2. Cell สุดท้ายจะเปิดหน้าต่างกล้อง
3. วางมือในกรอบสีเขียว
4. กด `q` เพื่อปิด

In [1]:
# 0. Install dependencies 
# pip install opencv-python tensorflow numpy 

In [2]:
# 0b. Download MediaPipe Hand Landmarker model (~2 MB) 
import urllib.request, os

MODEL_FILE = "hand_landmarker.task"
MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

if not os.path.exists(MODEL_FILE):
    print("Downloading hand_landmarker.task ...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)
    print("Download Successful!")
else:
    print(f"The {MODEL_FILE} file has been found. Skip this step.")

The hand_landmarker.task file has been found. Skip this step.


In [3]:
# 1. Imports 
import cv2
import numpy as np
import tensorflow as tf
import time

print(f"TensorFlow : {tf.__version__}")
print(f"OpenCV     : {cv2.__version__}")

TensorFlow : 2.21.0
OpenCV     : 4.13.0


In [4]:
# 2. Config
MODEL_PATH  = "outputs/baseline_cnn/best_model.keras"  
IMG_SIZE    = 28                            
CONFIDENCE  = 0.60                         # threshold ขั้นต่ำในการแสดงผล

# Sign Language MNIST ไม่มี J (9) และ Z (25)
CLASS_NAMES = [c for c in 'ABCDEFGHIKLMNOPQRSTUVWXY']  # 24 classes
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

Classes (24): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


In [5]:
# 3. Load Model 
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()
print("\nModel loaded successfully!")

Model: "Baseline_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         6,16

 Total params: 1,325,162 (5.06 MB)

 Trainable params: 441,336 (1.68 MB)

 Non-trainable params: 1,152 (4.50 KB)

 Optimizer params: 882,674 (3.37 MB)


Model loaded successfully!


In [6]:
#  4. Preprocessing Function 
def preprocess_roi(roi):
    # 1. Grayscale (เหมือนเดิม)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # 2. ✨ CLAHE — ปรับ contrast แบบ local (ช่วยมากเมื่อแสงไม่สม่ำเสมอ)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    gray = clahe.apply(gray)

    # 3. ✨ Gaussian blur เบาๆ — ลด noise จากกล้อง
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # 4. Resize 28×28 (เหมือนเดิม)
    resized = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)

    # 5. ✨ Normalize แบบ per-image (ดีกว่า /255 เมื่อ brightness ต่างกัน)
    resized = resized.astype(np.float32)
    mean, std = resized.mean(), resized.std()
    normalized = (resized - mean) / (std + 1e-6)
    # clip ให้อยู่ใน range [0, 1] เพื่อให้ตรงกับ training
    normalized = np.clip((normalized + 3) / 6, 0.0, 1.0)

    return normalized.reshape(1, 28, 28, 1)

def predict(roi):
    """
    ทำนายจาก ROI
    Returns: (label, confidence)
    """
    img = preprocess_roi(roi)
    probs = model.predict(img, verbose=0)[0]  # shape (24,)
    idx   = int(np.argmax(probs))
    conf  = float(probs[idx])
    return CLASS_NAMES[idx], conf


print("Functions ready")

Functions ready


In [ ]:
# ── 5. Real-Time Detection Loop (MediaPipe Tasks API) ──────
# Press  q  to quit
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from collections import deque, Counter

PADDING = 30  # padding around hand (pixels)

# ─── MediaPipe Hand Landmarker setup ───
base_options  = mp_python.BaseOptions(model_asset_path="hand_landmarker.task")
options       = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
hand_detector = mp_vision.HandLandmarker.create_from_options(options)

def find_hand_bbox(frame):
    """
    Detect hand using MediaPipe Tasks API.
    Returns: (x1, y1, x2, y2) or None if no hand found
    """
    h, w = frame.shape[:2]
    rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result   = hand_detector.detect(mp_image)

    if not result.hand_landmarks:
        return None

    landmarks = result.hand_landmarks[0]
    xs = [lm.x * w for lm in landmarks]
    ys = [lm.y * h for lm in landmarks]

    x_min = max(0, int(min(xs)) - PADDING)
    y_min = max(0, int(min(ys)) - PADDING)
    x_max = min(w, int(max(xs)) + PADDING)
    y_max = min(h, int(max(ys)) + PADDING)

    # Force square bounding box (model expects square input)
    side = max(x_max - x_min, y_max - y_min)
    cx   = (x_min + x_max) // 2
    cy   = (y_min + y_max) // 2

    x1 = max(0, cx - side // 2)
    y1 = max(0, cy - side // 2)
    x2 = min(w, x1 + side)
    y2 = min(h, y1 + side)

    return x1, y1, x2, y2


cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open camera. Try VideoCapture(1) or VideoCapture(2)")

prev_time  = time.time()
pred_queue = deque(maxlen=5)

print("📷 Camera ready — raise your hand | press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to read frame")
        break

    # Flip horizontally (mirror effect)
    frame = cv2.flip(frame, 1)
    h, w  = frame.shape[:2]

    bbox = find_hand_bbox(frame)

    if bbox is not None:
        x1, y1, x2, y2 = bbox
        roi = frame[y1:y2, x1:x2]

        if roi.size > 0:
            label, conf = predict(roi)

            if conf >= CONFIDENCE:
                pred_queue.append(label)

            smooth_label = Counter(pred_queue).most_common(1)[0][0] if pred_queue else "?"

            box_color = (0, 255, 0) if conf >= CONFIDENCE else (0, 165, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

            text_label = f"{smooth_label}  ({conf*100:.1f}%)"
            cv2.putText(frame, text_label,
                        (x1, max(y1 - 15, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                        box_color, 3, cv2.LINE_AA)

            # ROI preview (bottom-right corner)
            roi_preview = cv2.resize(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), (112, 112))
            roi_bgr = cv2.cvtColor(roi_preview, cv2.COLOR_GRAY2BGR)
            frame[h-120:h-8, w-120:w-8] = roi_bgr
            cv2.rectangle(frame, (w-121, h-121), (w-7, h-7), (100, 100, 100), 1)
    else:
        cv2.putText(frame, "No hand detected",
                    (w // 2 - 160, h // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (0, 0, 255), 2, cv2.LINE_AA)
        pred_queue.clear()

    # FPS counter
    now       = time.time()
    fps       = 1.0 / max(now - prev_time, 1e-6)
    prev_time = now
    cv2.putText(frame, f"FPS: {fps:.1f}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2, cv2.LINE_AA)
    cv2.putText(frame, "Press 'q' to quit",
                (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1, cv2.LINE_AA)

    cv2.imshow("Sign Language Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hand_detector.close()
print("\nCamera closed")

I0000 00:00:1776683702.674581 10782819 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776683702.690162 10782822 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776683702.710329 10782820 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


📷 Camera ready — raise your hand | press 'q' to quit


W0000 00:00:1776683704.213214 10782820 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.



Camera closed


: 